# Grad-CAM Validation & ROI Agreement

**Production pipeline this notebook implements and tests:**

```
full mammogram (ANY size) -> preprocess -> model -> Grad-CAM -> heatmap at the ORIGINAL size
```

The model takes **only the full image**. The cropped lesion patch and the ROI mask are
training-time signals; neither is needed here. The ROI mask appears in this notebook purely as
*ground truth to score the heatmap against* -- it is never fed to the model.

### The three questions this notebook answers

1. **Does the heatmap come back at the right size?** The model sees a 384x640 padded tensor, but a
   user hands us e.g. a 3024x5063 mammogram. The explanation must be overlayable on *their* image,
   so the padding has to be stripped and the map resized back. `cam_to_original` inverts the
   preprocessing geometry exactly. (Section 1.)
2. **Is the heatmap actually pointing at the lesion?** Measured, not eyeballed, on three metrics
   fixed in advance -- **IoU**, **Dice coefficient** and **Pointing Game accuracy** against the
   ground-truth ROI masks. (Sections 3-5.)
3. **Which of the four architectures explains itself best?** All four v4 control checkpoints
   (`densenet121`, `efficientnet_b2`, `resnet50`, `vgg16`) are scored on the same test images, then
   ranked with paired significance tests and a multiple-comparison correction. (Section 6.)

### The three headline metrics

| metric | question it answers | needs a threshold? |
|---|---|---|
| **IoU** | how much of the heatmap-vs-lesion *union* is overlap? | yes |
| **Dice** | same overlap, harmonic convention: `2*inter / (|pred| + |truth|)` | yes |
| **Pointing Game accuracy** | does the single hottest pixel land inside the lesion? | **no** |

Grad-CAM is continuous and the ROI is binary, so IoU and Dice only exist once the heatmap is
thresholded at some `tau` -- the threshold is what turns a heat value into the claim *"the model
says this pixel is lesion"*. It is fixed at the conventional `tau = 0.50` (half of each map's peak)
**before any result is seen**, and the appendix to section 5 shows the ranking is the same at every
threshold from 0.05 to 0.95. Pointing Game accuracy needs no threshold at all, which is exactly why
it is kept as the third metric: it cannot be tuned.

Supporting diagnostics (concentration ratio, pixel AP/AUROC, peak-in-padding) are still computed
and reported, but the three above are the ones the thesis quotes.

### Why measurement matters

Nearly every CBIS-DDSM paper that shows Grad-CAM shows three hand-picked heatmaps and asserts they
look right. A quantified, statistically tested localization result is a stronger contribution than
a couple of points of accuracy -- and unlike accuracy, it is largely within our control.

Ranking backbones by *interpretability* rather than accuracy is the part that is genuinely novel
here, and it is only meaningful because sections 1 and 3 first establish that the measurement
itself is sound: the geometry round-trips exactly, CAM peaks track a known input patch, and a
randomly-initialized model scores at chance on all three metrics.

### What runs

`MODE = "single"` loads the four trained hi-res single-input control checkpoints from
`models/v4_hires_control_gap/`. These are the deployable models, so the numbers below are the real
ones to quote.

A dual-input teacher path (`MODE = "teacher"`) is kept for reference only: it zeroes the crop
branch and runs far from the 224x224 it was trained at, so its scores land just above a random
model and must not appear in the thesis.

No seg-aux hi-res checkpoint exists yet, so section 7's control-vs-segaux comparison no-ops until
one is trained.

In [1]:
import os, sys, warnings


def _find_repo_root(markers=("models", "dataframes", "notebooks")):
    """Walk up from the CWD until all markers are found. os.path.abspath("../..") only works if
    the Jupyter kernel's working directory is exactly notebooks/explainability -- VS Code's
    Jupyter extension does not always start it there (it can default to the workspace root), which
    silently sends every path built from ROOT to the wrong place, models/ included."""
    d = os.path.abspath(os.getcwd())
    for _ in range(6):
        if all(os.path.isdir(os.path.join(d, m)) for m in markers):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    raise RuntimeError(f"could not locate the repo root by walking up from {os.getcwd()!r} -- "
                       "expected to find models/, dataframes/ and notebooks/ somewhere above it")


ROOT = _find_repo_root()
sys.path.insert(0, os.path.join(ROOT, "notebooks", "common"))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import importlib
import hires_lib as H

# hires_lib.py is edited alongside this notebook. Without the reload, a kernel that
# imported it earlier in the session keeps the OLD module object and any newly added
# function comes back as AttributeError -- which looks like a missing dependency rather
# than a stale cache.
H = importlib.reload(H)

warnings.filterwarnings("ignore", category=FutureWarning)
Image.MAX_IMAGE_PIXELS = None

PNG_ROOT = os.path.join(ROOT, "cbis_ddsm_png")
CKPT_DIR = os.path.join(ROOT, "notebooks", "experiment_1_baseline", "checkpoints")
NB_DIR = os.path.join(ROOT, "notebooks", "explainability")
RESULTS_DIR = os.path.join(NB_DIR, "results")
FIG_DIR = os.path.join(NB_DIR, "figures")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

H.set_seed()
print("kernel cwd:", os.getcwd())
print("repo root :", ROOT)
print("torch", torch.__version__, "| device", H.DEVICE)
print(f"cache {H.HIRES_W} x {H.HIRES_H} (W x H)")

kernel cwd: C:\Users\Drew\Documents\University\2025-2026\2nd Semester\Thesis\Experiment\notebooks\explainability
repo root : C:\Users\Drew\Documents\University\2025-2026\2nd Semester\Thesis\Experiment
torch 2.11.0+cu128 | device cuda
cache 384 x 640 (W x H)


In [2]:
test_df = pd.read_csv(os.path.join(ROOT, "dataframes", "test_df_hires.csv"))
val_df = pd.read_csv(os.path.join(ROOT, "dataframes", "val_df_hires.csv"))
print("test:", len(test_df), "| val:", len(val_df))

_BS = chr(92)


def winlong(p):
    """CBIS-DDSM's nested DICOM UID folders blow past Windows MAX_PATH."""
    p = os.path.abspath(p)
    return (_BS * 2 + "?" + _BS + p) if not p.startswith(_BS * 2) else p


def source_path(row):
    return os.path.join(PNG_ROOT, str(row["image file path"]).replace("/", os.sep))


_size_cache = {}


def source_size(row):
    """(W, H) of the ORIGINAL png. Cached -- reopening a 15MP png per call is slow."""
    p = source_path(row)
    if p not in _size_cache:
        with Image.open(winlong(p)) as im:
            _size_cache[p] = im.size
    return _size_cache[p]


print("example source size (W x H):", source_size(test_df.iloc[0]))

test: 319 | val: 317
example source size (W x H): (3552, 5944)


## 1. The production path

`explain(image_path)` is the deployable function: raw image in, probability + original-resolution
heatmap out. Everything else exists to test it.

`cam_to_original` inverts `fit_pad`. The geometry is fully determined by the source dimensions, so
no state has to be carried between preprocessing and postprocessing.

In [3]:
def preprocess(pil_gray):
    """Raw PIL image -> (1,3,640,384) normalized tensor, identical geometry to the cache builder,
    so a model trained on the cache sees exactly this at inference."""
    canvas = H.fit_pad(pil_gray, Image.LANCZOS)
    t = H.TF.to_tensor(canvas)
    return H.TF.normalize(t.expand(3, -1, -1).clone(), H.IMAGENET_MEAN, H.IMAGENET_STD).unsqueeze(0)


def explain(image_path, cam_fn, class_idx=1):
    """THE PRODUCTION CALL.

    Returns (prob_malignant, heatmap) where heatmap.shape == (orig_H, orig_W) -- the size of the
    image the caller supplied, NOT the 384x640 preprocessed size.
    """
    with Image.open(winlong(image_path)) as im:
        pil = im.convert("L")
        ow, oh = pil.size
        x = preprocess(pil).to(H.DEVICE)
    cam, probs = cam_fn(x, class_idx=class_idx)
    return float(probs[0, 1]), H.cam_to_original(cam[0], ow, oh)


print("explain() returns a heatmap at the ORIGINAL image resolution.")

explain() returns a heatmap at the ORIGINAL image resolution.


### Round-trip check

A synthetic marker at a known fraction of the source must return to that same fraction after
`fit_pad` -> `cam_to_original`, across the real aspect ratios in the dataset.

In [4]:
print(f"{'source (WxH)':>18} | {'padded core':>13} | {'returned (WxH)':>15} | rel. error (x, y)")
worst = 0.0
for (ow, oh) in [(2986, 5356), (3024, 5063), (4144, 6736), (2041, 4384), (3552, 5944)]:
    nw, nh, ox, oy = H.fit_pad_geometry(ow, oh)
    cam = np.zeros((H.HIRES_H, H.HIRES_W), np.float32)
    cy, cx = oy + int(nh * 0.35), ox + int(nw * 0.65)
    yy, xx = np.ogrid[:H.HIRES_H, :H.HIRES_W]
    cam[(yy - cy) ** 2 + (xx - cx) ** 2 <= 8 ** 2] = 1.0
    back = H.cam_to_original(cam, ow, oh)
    py, px = np.unravel_index(np.argmax(back), back.shape)
    ey, ex = abs(py / oh - 0.35), abs(px / ow - 0.65)
    worst = max(worst, ey, ex)
    print(f"{ow:8d}x{oh:<8d} | {nw:4d}x{nh:<7d} | {back.shape[1]:6d}x{back.shape[0]:<7d} | "
          f"({ex:.4f}, {ey:.4f})")
assert worst < 0.02, f"round-trip error {worst:.4f} too large"
print(f"\nworst relative error {worst:.4f} -- output size matches input size exactly")

      source (WxH) |   padded core |  returned (WxH) | rel. error (x, y)
    2986x5356     |  357x640     |   2986x5356    | (0.0070, 0.0102)


    3024x5063     |  382x640     |   3024x5063    | (0.0075, 0.0101)


    4144x6736     |  384x624     |   4144x6736    | (0.0081, 0.0111)
    2041x4384     |  298x640     |   2041x4384    | (0.0106, 0.0101)


    3552x5944     |  382x640     |   3552x5944    | (0.0073, 0.0102)

worst relative error 0.0111 -- output size matches input size exactly


## 2. Load the models

All four v4 control checkpoints (`densenet121`, `efficientnet_b2`, `resnet50`, `vgg16`) from
`models/v4_hires_control_gap/`, trained identically -- same 640x384 cache, same GAP pooling, same
alpha, same seed 42, same batch size. Only the backbone differs, which is what makes the
architecture comparison in section 6 a fair one.

Every section below loops over `ARCHS`. `ARCH` is only the default argument for `load_model()`.

In [5]:
MODE = "single"          # "teacher" | "single"
ARCHS = ["densenet121", "efficientnet_b2", "resnet50", "vgg16"]
ARCH = ARCHS[0]          # default for load_model(); no section depends on it being densenet

# Colour is bound to the ARCHITECTURE, not to its rank, so a backbone keeps its hue in every
# figure and re-sorting never repaints the others. Checked for colourblind separation: worst
# adjacent pair dE 9.1 under protanopia. (The green/red pair this notebook previously used for
# the class split is dE 1.2 under deuteranopia -- one colour to those readers. Replaced.)
ARCH_COLOR = dict(zip(ARCHS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]))
CLASS_COLOR = {1: "#eb6834", 0: "#2a78d6"}      # malignant / benign
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"

V4_DIR = os.path.join(ROOT, "models", "v4_hires_control_gap")


def single_ckpt_path(arch):
    return os.path.join(V4_DIR, f"hires_control_gap_a0.5_{arch}.pth")


class TeacherFullBranch(torch.nn.Module):
    """Exposes the teacher's full-image branch with a SingleInputModel-shaped interface so the
    identical Grad-CAM code runs on both. The crop branch is zero-filled: we are explaining what
    the full-image pathway responds to, and the deployed model has no crop at all."""

    def __init__(self, teacher):
        super().__init__()
        self.teacher = teacher
        # drop the trailing AdaptiveAvgPool2d so `extractor` yields a SPATIAL map
        self.extractor = torch.nn.Sequential(*[m for m in teacher.full_extractor
                                               if not isinstance(m, torch.nn.AdaptiveAvgPool2d)])

    def forward(self, x):
        spatial = self.extractor(x)
        f = torch.flatten(torch.nn.functional.adaptive_avg_pool2d(spatial, 1), 1)
        return self.teacher.classifier(torch.cat([f, torch.zeros_like(f)], dim=1))


def load_model(arch=None, mode=None, ckpt=None):
    """Parameterized so every section can reload once per architecture. The pooling/use_seg flags
    come from the checkpoint, never from a notebook constant -- a mismatch there loads without
    error and silently changes what is being explained."""
    arch = arch or ARCH
    mode = mode or MODE
    if mode == "teacher":
        m = TeacherFullBranch(H.load_teacher(arch, CKPT_DIR)).to(H.DEVICE).eval()
        return m, f"teacher full-branch ({arch})"
    ckpt = ckpt or single_ckpt_path(arch)
    ck = torch.load(ckpt, map_location=H.DEVICE, weights_only=False)
    assert list(ck.get("input_size", [])) == [H.HIRES_H, H.HIRES_W], (
        f"checkpoint input_size {ck.get('input_size')} != {[H.HIRES_H, H.HIRES_W]}. A 224-trained "
        "state dict loads without error at 640x384 and is silently wrong.")
    assert ck.get("arch", arch) == arch, f"checkpoint arch {ck.get('arch')} != requested {arch}"
    m = H.SingleInputModel(arch, pooling=ck.get("pooling", "gap"),
                           use_seg=ck.get("use_seg", False)).to(H.DEVICE)
    m.load_state_dict(ck["model_state_dict"])
    m.eval()
    return m, f"{arch} ({os.path.basename(ckpt)})"


# --- preflight: every checkpoint loads, and what explanation resolution does each buy? ---
# Failing here costs seconds; failing inside section 5 costs the whole sweep.
grid_rows = []
for arch in ARCHS:
    m, _ = load_model(arch=arch)
    with torch.no_grad():
        probe = m.extractor(torch.zeros(1, 3, H.HIRES_H, H.HIRES_W, device=H.DEVICE))
    gh, gw = int(probe.shape[-2]), int(probe.shape[-1])
    grid_rows.append({"arch": arch, "feat_ch": int(probe.shape[1]), "cam_grid": f"{gh}x{gw}",
                      "cells": gh * gw,
                      "cell_px": f"{H.HIRES_H // gh}x{H.HIRES_W // gw}"})
    del probe, m
    H.free_gpu()

GRID_TBL = pd.DataFrame(grid_rows)
print(GRID_TBL.to_string(index=False))
print("\ncam_grid IS the resolution of the explanation -- an upper bound on how precisely an")
print("architecture can point, before any question of whether it points correctly.")

           arch  feat_ch cam_grid  cells cell_px
    densenet121     1024    20x12    240   32x32
efficientnet_b2     1408    20x12    240   32x32
       resnet50     2048    20x12    240   32x32
          vgg16      512    20x12    240   32x32

cam_grid IS the resolution of the explanation -- an upper bound on how precisely an
architecture can point, before any question of whether it points correctly.


**That grid is the resolution of the explanation, and it is an architecture property.** On the old
224 cache it was 7x7 with the median lesion at 19x10 px -- smaller than one cell, so Grad-CAM could
not have localized a lesion even in principle. At 640x384 each of these backbones downsamples by
32, giving a 20x12 grid with the lesion at 35x34 px: about one cell. Coarse, but now genuinely
capable of pointing.

Because all four share that grid, none of them has a built-in resolution advantage -- any
difference in the localization scores below comes from what the backbone *learned*, not from how
finely it can express an answer.

## 3. Correctness checks before trusting any number

A localization metric that scores high for a random model is measuring the *dataset*, not the
model. These run first so the numbers that follow mean something.

Check (a) establishes the **chance floor for all three headline metrics** and is architecture-
specific (a random model's spatial statistics depend on its conv stem), so it runs once per
checkpoint in `ARCHS`. Check (b) tests the CAM-upsampling geometry itself, which is the same code
path regardless of backbone, so it runs once against a synthetic network.

In [ ]:
# --- shared helpers, and the threshold the overlap metrics are reported at ---
# IoU and Dice compare two BINARY sets, but Grad-CAM is continuous. tau is what turns "how hot is
# this pixel" into "does the model CLAIM this pixel": the model's answer is {pixels >= tau}, and
# that set is what gets compared against the ROI. Without a tau there is no set and no IoU.
#
# TAU = 0.50 -- half of each map's peak -- is the Grad-CAM/WSOL convention, and it is fixed HERE,
# before any result is seen. That is the property that matters: a threshold chosen after looking at
# test scores would be reporting the best of many tries. The appendix after step 3 sweeps every
# threshold and shows the ranking does not depend on this choice.
#
# GradCAM normalizes every map by its own max, so tau is a fraction of that image's peak and means
# the same thing on every image and every backbone.
TAU = 0.50
TAUS = np.round(np.arange(0.05, 0.96, 0.05), 2)    # appendix sweep only


def load_mask(row):
    return np.array(Image.open(row[H.ROI_HIRES_COL]).convert("L")) > 127


def tensor_from_cached(row):
    t = H.TF.to_tensor(Image.open(row[H.FULL_HIRES_COL]).convert("L"))
    return H.TF.normalize(t.expand(3, -1, -1).clone(), H.IMAGENET_MEAN, H.IMAGENET_STD).unsqueeze(0)


def cam_for_row(row, cam_fn, class_idx=1):
    cam, probs = cam_fn(tensor_from_cached(row).to(H.DEVICE), class_idx=class_idx)
    return cam[0], float(probs[0, 1])


# --- (a) a random-init model must score at chance on all three headline metrics ---
# Scored at the SAME tau the trained models are scored at, so the two tables are directly
# comparable. `best_*` additionally lets the random model choose its threshold in hindsight, which
# no trained model is allowed to do -- a trained model that cannot beat that deliberately
# handicapped-in-its-favour number is not distinguishable from noise.
sub = test_df.head(40)
chance, rand_rows = [], []
for arch in ARCHS:
    rand_model = H.SingleInputModel(arch, pooling="gap", use_seg=False).to(H.DEVICE).eval()
    hits, ious, dices = [], [], []
    with H.GradCAM(rand_model) as cf:
        for _, row in tqdm(sub.iterrows(), total=len(sub), desc=f"random-init ({arch})", leave=False):
            cam, _ = cam_for_row(row, cf)
            mask, v = load_mask(row), H.validity_mask(*source_size(row))
            if not mask.any():
                continue
            m = H.localization_metrics(cam, mask, v)
            sw = H.overlap_sweep(cam, mask, v, TAUS)
            hits.append(m["pointing_hit"])
            ious.append(sw["iou"])
            dices.append(sw["dice"])
            chance.append(m["chance_rate"])
    del rand_model
    H.free_gpu()
    mi, md = np.nanmean(ious, axis=0), np.nanmean(dices, axis=0)
    at_tau, k = int(np.argmin(np.abs(TAUS - TAU))), int(np.nanargmax(md))
    rand_rows.append({"arch": arch, "PointingAcc_%": 100 * np.mean(hits),
                      "IoU@tau": mi[at_tau], "Dice@tau": md[at_tau],
                      "best_IoU": mi[k], "best_Dice": md[k], "best_at_tau": TAUS[k]})

RAND_TBL = pd.DataFrame(rand_rows)
CHANCE_PGA = 100 * float(np.mean(chance))
print(f"CHANCE FLOOR -- randomly initialized weights, IoU/Dice at the same tau = {TAU:.2f}")
print(RAND_TBL.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"\nThe lesion covers {CHANCE_PGA:.2f}% of valid pixels, so blind guessing scores ~2.5%")
print("pointing accuracy. Any trained model that does not clear this table is not localizing,")
print("whatever its classification accuracy says.")

In [ ]:
# --- (b) spatial correspondence against a known answer ---
# NOT an hflip-equivariance test: convolutions are translation-equivariant but NOT reflection-
# equivariant, so CAM(flip(x)) != flip(CAM(x)) even for perfectly correct code. Instead a stride-32
# conv maps input block [32i:32i+32, 32j:32j+32] to cell (i,j) exactly, so a bright patch at a
# known place MUST produce a CAM peak there. This also pins down H/W ordering in the upsample.
class _StrideNet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.extractor = torch.nn.Conv2d(3, 8, 32, stride=32, bias=False)
        torch.nn.init.constant_(self.extractor.weight, 0.01)
        self.classifier = torch.nn.Linear(8, 2)
        torch.nn.init.constant_(self.classifier.weight, 0.5)
        torch.nn.init.zeros_(self.classifier.bias)

    def forward(self, x):
        a = self.extractor(x)
        return self.classifier(torch.flatten(torch.nn.functional.adaptive_avg_pool2d(a, 1), 1))


snet = _StrideNet().to(H.DEVICE).eval()
for (ry, rx) in [(0.25, 0.75), (0.80, 0.20), (0.50, 0.50)]:
    xin = torch.zeros(1, 3, H.HIRES_H, H.HIRES_W, device=H.DEVICE)
    ty, tx = int(H.HIRES_H * ry), int(H.HIRES_W * rx)
    xin[0, :, ty:ty + 32, tx:tx + 32] = 5.0
    with H.GradCAM(snet) as cf:
        c, _ = cf(xin)
    py, px = np.unravel_index(np.argmax(c[0]), c[0].shape)
    print(f"  patch (y={ty:3d}, x={tx:3d}) -> CAM peak (y={py:3d}, x={px:3d})   "
          f"err ({abs(py - ty - 16)}, {abs(px - tx - 16)}) px")
    assert abs(py - ty - 16) <= 32 and abs(px - tx - 16) <= 32, "spatial mapping is wrong"
del snet
H.free_gpu()
print("\nCAM peaks track the driving input region in BOTH axes -- no transposition.")

## 4. Overlay: original image + ROI + Grad-CAM

All at **original resolution** -- what a radiologist would actually be shown. The green contour is
ground truth; the heatmap is the model's explanation. One grid is produced per architecture in
`ARCHS`, on the same fixed set of images, so the four are directly comparable. Whether they agree
is answered numerically in section 5.

In [ ]:
DISP_DIV = 6      # display downscale; a full-res float heatmap per panel would exhaust host RAM


def overlay_panel(row, cam_fn, ax_row, show_titles=False):
    src = source_path(row)
    with Image.open(winlong(src)) as im:
        pil = im.convert("L")
        ow, oh = pil.size
        dw, dh = ow // DISP_DIV, oh // DISP_DIV
        disp = np.asarray(pil.resize((dw, dh), Image.BILINEAR), np.float32) / 255.0

    prob, heat_full = explain(src, cam_fn)
    heat = np.asarray(Image.fromarray(heat_full, mode="F").resize((dw, dh), Image.BILINEAR),
                      np.float32)
    del heat_full                       # ~61MB at full res -- release immediately

    # the cached mask lives in PADDED space; strip the padding before display
    nw, nh, ox, oy = H.fit_pad_geometry(ow, oh)
    core = load_mask(row)[oy:oy + nh, ox:ox + nw].astype(np.float32)
    mask_d = np.asarray(Image.fromarray(core, mode="F").resize((dw, dh), Image.NEAREST)) > 0.5

    truth = "Malignant" if H.encode_label(row[H.LABEL_COL]) == 1 else "Benign"
    ax_row[0].imshow(disp, cmap="gray")
    ax_row[1].imshow(disp, cmap="gray")
    ax_row[1].contour(mask_d, [0.5], colors="lime", linewidths=1.4)
    ax_row[2].imshow(disp, cmap="gray")
    ax_row[2].imshow(heat, cmap="jet", alpha=0.45)
    ax_row[3].imshow(disp, cmap="gray")
    ax_row[3].imshow(heat, cmap="jet", alpha=0.40)
    ax_row[3].contour(mask_d, [0.5], colors="lime", linewidths=1.4)
    if show_titles:
        for a, t in zip(ax_row, ["original", "+ ROI (truth)", "+ Grad-CAM", "both"]):
            a.set_title(t, fontsize=10)
    ax_row[0].set_ylabel(f"{truth}\np(malig)={prob:.2f}", fontsize=8)
    for a in ax_row:
        a.set_xticks([])
        a.set_yticks([])


# Random draw, not .head() -- the first rows of test_df_hires.csv are whatever order the CSV
# happened to be written in, not a representative sample. Seeded on H.SEED so the same five
# images come up on every rerun. Restricted to rows with a non-empty ROI mask, or the "+ ROI"
# panel would show no contour at all.
num = 10
H.SEED = num
batch_num = num
N_MALIGNANT, N_BENIGN = 3, 2
lab = test_df[H.LABEL_COL].apply(H.encode_label)
has_mask = test_df.apply(lambda r: load_mask(r).any(), axis=1)
pool_mal = test_df[(lab == 1) & has_mask]
pool_ben = test_df[(lab == 0) & has_mask]
show = pd.concat([
    pool_mal.sample(n=min(N_MALIGNANT, len(pool_mal)), random_state=H.SEED),
    pool_ben.sample(n=min(N_BENIGN, len(pool_ben)), random_state=H.SEED),
]).reset_index(drop=True)
print(f"showing {len(show)} images (seed={H.SEED}): "
      f"{show[H.LABEL_COL].apply(H.encode_label).map({1: 'malignant', 0: 'benign'}).tolist()}")

for arch in ARCHS:
    m, note = load_model(arch=arch)
    fig, axes = plt.subplots(len(show), 4, figsize=(13, 3.1 * len(show)))
    with H.GradCAM(m) as cf:
        for i, (_, row) in enumerate(show.iterrows()):
            overlay_panel(row, cf, axes[i], show_titles=(i == 0))
    plt.suptitle(f"Grad-CAM vs ground-truth ROI  -  {note}", y=1.002)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f"batch_{batch_num}",f"gradcam_overlay_grid_{arch}.png"), dpi=110,
                bbox_inches="tight")
    plt.show()
    del m
    H.free_gpu()

## 5. The three localization metrics

This section produces the numbers the thesis quotes: **IoU**, **Dice coefficient**, and **Pointing
Game accuracy**, for each of the four architectures, on the same test images.

### What each metric is

Write `P` for the set of pixels the heatmap claims and `G` for the ground-truth ROI. Both are
restricted to `valid` -- the non-padding region of the 384x640 canvas, because 7-12% of this cache
is black padding and a hit there is meaningless.

```
IoU   =  |P & G| / |P | G|
Dice  =  2|P & G| / (|P| + |G|)
PGA   =  fraction of images whose hottest valid pixel falls inside G
```

### What `tau` is for

Grad-CAM returns a continuous heat value per pixel; the ROI is binary. `P` does not exist until you
say *how hot counts as claimed* -- that is `tau`, and `P = {pixels with cam >= tau}`. It is not a
tuning knob bolted on afterwards; it is part of the definition of the metric.

The trade-off it controls: a low `tau` claims a large sloppy region (the union blows up, IoU
falls), a high `tau` claims only the few pixels around the peak (the intersection shrinks, IoU
falls again). Somewhere between is the honest reading of the heatmap.

**`tau = 0.50` is fixed in advance**, at half of each map's peak -- the Grad-CAM/WSOL convention.
The thing that would be dishonest is choosing it *after* seeing test scores; a conventional value
fixed up front avoids that completely, and the appendix confirms the ranking is the same at every
threshold from 0.05 to 0.95.

**Pointing Game accuracy needs no `tau` at all** -- it looks only at the argmax. That is why it is
the third metric: it is the one that cannot be tuned, and the only one that is genuinely
independent of the other two.

### One caveat to carry into the write-up

`Dice = 2*IoU/(1+IoU)` exactly, because `|P|+|G| = |P|G| + |P&G|`. Per image, IoU and Dice always
agree on which model is better -- they are one measurement in two conventions. Both are reported
because both are conventional, but they are **one** piece of evidence, not two. Step 1 asserts the
identity rather than leaving it to be assumed.

### The three steps

| step | what it does |
|---|---|
| 1 | per-image scorer: pointing game + IoU/Dice, plus a self-test |
| 2 | run it over the test split for all four checkpoints |
| 3 | the three metrics per architecture, with bootstrap 95% CIs |

Everything is computed in 384x640 space where the mask already lives -- mathematically identical to
computing at original resolution and vastly cheaper than materializing a 61 MB float array per
image.

### Step 1 -- the per-image scorer, and a self-test on it

One row per image: the pointing game (no threshold), IoU and Dice at `tau`, and the threshold-free
diagnostics. The full `tau` grid is swept too -- it costs nothing on top of the same forward pass
and it is what the appendix plots.

`H.overlap_sweep` replaces "threshold a 640x384 array, 19 times, per image" with one `argsort` plus
a vectorized `searchsorted`. That is a real optimization with real off-by-one risk, and a silent
disagreement would corrupt every IoU and Dice below -- so it is checked against the obvious
implementation instead of trusted.

```python
def evaluate_localization(df, cam_fn, ...):
    for each image with a non-empty ROI:
        cam  = GradCAM(model, image)          # (640, 384) in [0, 1], peak exactly 1.0
        mask = ground-truth ROI               # (640, 384) bool
        v    = validity_mask(...)             # (640, 384) bool -- excludes padding
        pointing_hit = mask[argmax(cam * v)]
        iou, dice    = overlap(cam >= TAU, mask, v)
```

In [ ]:
# --- Step 1: the per-image scorer ---
def col(metric, tau):
    """Column name for a swept metric. One helper so `iou@0.85` is never spelled out by hand."""
    return f"{metric}@{tau:.2f}"


def evaluate_localization(df, cam_fn, class_idx=1, limit=None, desc="localization"):
    """One row per image with a non-empty ROI: pointing game, the IoU/Dice sweep over TAUS, and the
    threshold-free diagnostics.

    `row_idx` records which source row each result came from. The skip condition depends only on
    the mask, so every architecture yields the same rows in the same order -- but section 6 runs
    PAIRED tests on these frames, so step 2 asserts that rather than assuming it.
    """
    rows = []
    it = df if limit is None else df.head(limit)
    for i, row in tqdm(it.iterrows(), total=len(it), desc=desc, leave=False):
        mask = load_mask(row)
        if not mask.any():
            continue
        cam, prob = cam_for_row(row, cam_fn, class_idx)      # (640,384) in [0,1], peak exactly 1.0
        v = H.validity_mask(*source_size(row))               # excludes the padding
        m = H.localization_metrics(cam, mask, v)             # pointing game + threshold-free extras
        sw = H.overlap_sweep(cam, mask, v, TAUS)             # IoU and Dice at every tau, one pass
        for t, iou, dice in zip(sw["tau"], sw["iou"], sw["dice"]):
            m[col("iou", t)], m[col("dice", t)] = iou, dice
        m.update(row_idx=i, prob_malignant=prob, label=H.encode_label(row[H.LABEL_COL]))
        rows.append(m)
    return pd.DataFrame(rows)


def summarize(d, name, tau=None):
    """Compact per-model row at a threshold -- used by section 7's paired comparison."""
    tau = TAU if tau is None else tau
    return {"model": name, "n": len(d), "tau": tau,
            "IoU": d[col("iou", tau)].mean(),
            "Dice": d[col("dice", tau)].mean(),
            "PointingAcc_%": 100 * d["pointing_hit"].mean(),
            "PGA@15px_%": 100 * d["pointing_hit_tol"].mean(),
            "conc_ratio_x": d["concentration_ratio"].mean(),
            "pixel_ap": d["pixel_ap"].mean(),
            "peak_in_padding_%": 100 * d["peak_in_padding"].mean(),
            "median_peak_dist_px": d["peak_dist_px"].median()}


# --- self-test: the fast sweep must equal the naive per-tau implementation ---
# Run on a REAL mask and validity region (so the padding and the lesion geometry are the real
# thing) with a random CAM, which exercises every threshold densely. A silent off-by-one in
# searchsorted would corrupt every IoU and Dice in this notebook.
_row0 = test_df.iloc[0]
_m0, _v0 = load_mask(_row0), H.validity_mask(*source_size(_row0))
_cam0 = np.random.default_rng(0).random((H.HIRES_H, H.HIRES_W))
_sw = H.overlap_sweep(_cam0, _m0, _v0, TAUS)
for _t, _i, _d in zip(_sw["tau"], _sw["iou"], _sw["dice"]):
    assert np.isclose(_i, H.iou_at(_cam0, _m0, _v0, _t), equal_nan=True), f"IoU differs at {_t}"
    assert np.isclose(_d, H.dice_at(_cam0, _m0, _v0, _t), equal_nan=True), f"Dice differs at {_t}"
assert np.allclose(_sw["dice"], 2 * _sw["iou"] / (1 + _sw["iou"]), equal_nan=True)

print(f"tau grid ({len(TAUS)} values): {TAUS}")
print(f"overlap_sweep matches iou_at/dice_at at all {len(TAUS)} thresholds,")
print("and Dice == 2*IoU/(1+IoU) holds exactly -- the two are one metric in two conventions.")

### Step 2 -- run the scorer over the test split, for all four checkpoints

Four passes, ~319 images each. Only the test split is scored: with `tau` fixed by convention there
is nothing to select on a validation set, so the val pass that an adaptive threshold would have
required is simply not needed.

In [ ]:
# --- Step 2: score every architecture on the test split ---
ALL_LOC = {}
for arch in ARCHS:
    m, note = load_model(arch=arch)
    with H.GradCAM(m) as cf:
        ALL_LOC[arch] = evaluate_localization(test_df, cf, desc=f"test ({arch})")
    ALL_LOC[arch].to_csv(os.path.join(RESULTS_DIR, f"localization_{MODE}_{arch}.csv"), index=False)
    del m
    H.free_gpu()

# Every architecture must have scored the SAME images in the SAME order, or section 6's paired
# tests silently compare unrelated pairs. The skip condition only depends on the mask, so this
# holds by construction -- which is exactly the kind of assumption worth asserting.
_ref = ALL_LOC[ARCHS[0]]["row_idx"].tolist()
for arch in ARCHS[1:]:
    assert ALL_LOC[arch]["row_idx"].tolist() == _ref, f"{arch} evaluated a different test row set"

print(f"test : {len(_ref)} of {len(test_df)} images had a non-empty ROI mask")
print(f"all {len(ARCHS)} architectures scored identical image sets in identical order -- paired.")

### Step 3 -- the three metrics

`tau` was fixed before anything ran, so the test split is simply read. This is the table the thesis
quotes: **IoU, Dice and Pointing Game accuracy per architecture, each with a bootstrap 95% CI.**

Two things to expect, and neither is a bug:

- **The absolute IoU/Dice values are low.** The lesion covers ~2.5% of valid pixels while a
  Grad-CAM blob on a 20x12 grid cannot be smaller than one cell (~32x32 px). The metric is
  penalizing a resolution limit as much as a localization error -- which is why the value belongs
  next to `cam_grid` from section 2, and why the threshold-free pointing game is reported beside it.
- **Pointing Game accuracy is much higher than IoU.** It asks a strictly easier question -- one
  pixel in the right place, versus the whole region being the right shape.

`peak_in_pad_%` stays in the table as a fraud check: if the model were keying on canvas geometry
rather than tissue, its hottest pixel would land in the black padding.

In [ ]:
# --- Step 3: the three metrics at the fixed tau ---
_rng = np.random.default_rng(H.SEED)


def boot_ci(values, n_boot=2000):
    """Percentile bootstrap 95% CI of the mean. At n~319 the SE on a proportion is ~2.6pp, so a
    bare point estimate is not defensible and every headline number carries one."""
    v = np.asarray(values, dtype=float)
    v = v[~np.isnan(v)]
    if v.size == 0:
        return float("nan"), float("nan")
    b = _rng.choice(v, size=(n_boot, v.size), replace=True).mean(axis=1)
    return float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))


head_rows = []
for arch in ARCHS:
    d = ALL_LOC[arch]
    iou, dice = d[col("iou", TAU)], d[col("dice", TAU)]
    pga = d["pointing_hit"].astype(float)
    (il, ih), (dl, dh), (pl, ph) = boot_ci(iou), boot_ci(dice), boot_ci(pga)
    head_rows.append({
        "arch": arch, "n": len(d), "tau": TAU,
        "IoU": iou.mean(), "IoU_95CI": f"[{il:.3f}, {ih:.3f}]",
        "Dice": dice.mean(), "Dice_95CI": f"[{dl:.3f}, {dh:.3f}]",
        "PointingAcc_%": 100 * pga.mean(), "PGA_95CI_%": f"[{100 * pl:.1f}, {100 * ph:.1f}]",
        "PGA@15px_%": 100 * d["pointing_hit_tol"].mean(),
        "conc_ratio_x": d["concentration_ratio"].mean(),
        "peak_in_pad_%": 100 * d["peak_in_padding"].mean(),
    })

pd.set_option("display.width", 220)
HEADLINE = pd.DataFrame(head_rows).sort_values("Dice", ascending=False).reset_index(drop=True)
print(f"HEADLINE -- test split, tau = {TAU:.2f}")
print(HEADLINE.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
HEADLINE.to_csv(os.path.join(RESULTS_DIR, f"localization_headline_{MODE}_all_archs.csv"),
                index=False)

# The identity from the section header, now on the real means. f(x) = 2x/(1+x) is CONCAVE, so
# Jensen runs in exactly one direction: mean(Dice) <= f(mean(IoU)). A violation would mean the two
# columns were read at different thresholds.
print("\nsanity: f(x) = 2x/(1+x) is concave, so Jensen forces mean(Dice) <= f(mean(IoU)).")
for r in HEADLINE.itertuples():
    j = 2 * r.IoU / (1 + r.IoU)
    print(f"  {r.arch:16s} mean Dice {r.Dice:.4f} <= f(mean IoU) {j:.4f}   gap {j - r.Dice:.4f}  "
          f"{'ok' if r.Dice <= j + 1e-9 else 'VIOLATED'}")

# --- how far above its own random-init floor is each backbone? ---
# The headline table alone cannot show this, and it is the difference between "localizes" and
# "scores non-zero". Compared at the same tau, so the ratio is meaningful.
print(f"\nLIFT OVER THE SAME ARCHITECTURE'S RANDOM-INIT FLOOR (tau = {TAU:.2f})")
floor = RAND_TBL.set_index("arch")
for _, r in HEADLINE.iterrows():          # label access: 'PointingAcc_%' is not a valid attribute
    a = r["arch"]
    f_d, f_p = floor.loc[a, "Dice@tau"], floor.loc[a, "PointingAcc_%"]
    print(f"  {a:16s} Dice {f_d:.4f} -> {r['Dice']:.4f} "
          f"({r['Dice'] / max(f_d, 1e-9):5.1f}x)   "
          f"pointing {f_p:4.1f}% -> {r['PointingAcc_%']:4.1f}%")

### Appendix -- does the choice of `tau` change the answer?

`tau = 0.50` was fixed by convention before anything ran, which removes the selection problem but
raises a fair question: would a different threshold have produced a different ranking? This sweeps
all 19 and plots the answer.

If the curves keep their vertical order across the whole range, the conclusion is a property of the
models and not of the threshold, and the single number in step 3 can be quoted without
qualification. Where they cross, the affected architectures should be reported as tied.

In [ ]:
# --- Appendix: the ranking under every threshold ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4), sharex=True)

for ax, met, lbl in zip(axes, ["iou", "dice"], ["IoU", "Dice"]):
    for arch in ARCHS:
        y = np.array([ALL_LOC[arch][col(met, t)].mean() for t in TAUS])
        k = int(np.argmin(np.abs(TAUS - TAU)))
        ax.plot(TAUS, y, color=ARCH_COLOR[arch], lw=2.0, label=arch, zorder=2)
        ax.scatter([TAU], [y[k]], s=54, color=ARCH_COLOR[arch], zorder=4,
                   edgecolor="white", linewidth=1.8)
        # direct label: identity and value never rest on colour alone
        ax.annotate(f"{arch} {y[k]:.3f}", (TAU, y[k]), textcoords="offset points",
                    xytext=(10, 0), fontsize=8.2, color=INK, va="center")
    ax.axvline(TAU, color=MUTED, ls="--", lw=1.2, zorder=1)
    ax.set_xlabel("threshold tau (fraction of each map's peak)", color=MUTED, fontsize=9)
    ax.set_ylabel(f"mean {lbl}", color=MUTED, fontsize=9)
    ax.set_title(f"{lbl} vs threshold  -  dashed line: the reported tau = {TAU:.2f}",
                 fontsize=10, loc="left")
    ax.grid(color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.margins(x=0.18)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(GRID)
    ax.tick_params(colors=MUTED, labelsize=8.5)
axes[0].legend(frameon=False, fontsize=8.5, loc="upper right", labelcolor=INK)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "threshold_sweep_iou_dice.png"), dpi=150, bbox_inches="tight")
plt.show()

# State it numerically rather than leaving it to the eye: at how many of the 19 thresholds does
# each architecture hold its rank, and is the winner ever displaced?
order_at = {}
for t in TAUS:
    means = {a: ALL_LOC[a][col("dice", t)].mean() for a in ARCHS}
    order_at[t] = sorted(ARCHS, key=lambda a: -means[a])
winners = {t: o[0] for t, o in order_at.items()}
ref = order_at[TAU]
same = sum(1 for t in TAUS if order_at[t] == ref)
print(f"reported ordering at tau={TAU:.2f}: {' > '.join(ref)}")
print(f"identical ordering at {same} of {len(TAUS)} thresholds")
print(f"winner across all thresholds: {sorted(set(winners.values()))}")
for t in TAUS:
    if order_at[t] != ref:
        print(f"  tau={t:.2f}: {' > '.join(order_at[t])}")

### Reading the three numbers

**Read them against the floors, not against 1.0.** Section 3(a) gives the random-init score for
each architecture at the same `tau`, and the lesion-coverage rate gives the blind-guess pointing
accuracy (~2.5%). Step 3 prints the lift over that floor per architecture. A model near either
floor is not localizing, whatever its AUC says.

**IoU and Dice will look small, and the reason is geometric.** With the lesion at ~2.5% of valid
pixels and the CAM quantized to a 20x12 grid, even a perfectly-placed single-cell blob caps out at
a modest IoU. That is why the absolute value should be reported next to the `cam_grid` from section
2, and why the `tau` curve belongs in the appendix -- a single IoU invites the reader to compare it
against segmentation IoUs from models that were actually trained to segment.

**Pointing Game accuracy is the one to lead with in text.** No threshold, no tuning, and one
sentence of interpretation: *"the hottest point of the explanation falls inside the radiologist's
ROI on N% of test mammograms, against a 2.5% chance rate."*

What the harness has established by this point, independent of any architecture: sizes round-trip
exactly (section 1), CAM peaks track a known input patch in both axes (section 3b), a random model
scores at chance (section 3a), and the fast sweep equals the naive one (step 1). So differences
between the four backbones are real differences in what they learned, not artifacts of measurement.

Eyeballing which column is largest is still not enough -- with n~319 and heavily skewed per-image
overlaps, adjacent architectures can differ a lot in the mean and not at all in reality. Section 6
tests it.

## 6. Which architecture is the most interpretable?

All four were evaluated on the **same test images in the same order** (asserted in step 2), so this
is a paired comparison and paired tests apply -- far more powerful than checking whether bootstrap
CIs overlap, because per-image difficulty cancels out.

**The right test per metric.** Dice and IoU are continuous, bounded below at 0 and long-tailed, so
they get **Wilcoxon signed-rank**, not a t-test -- normality is not a safe assumption at any n.
Pointing Game accuracy is a paired *binary* outcome, so it gets **McNemar's exact test**, which
looks only at the images where the two models disagree. Using Wilcoxon on a 0/1 column would be
answering the right question with the wrong machinery.

**Holm-Bonferroni correction.** Four architectures means six pairwise tests. Uncorrected, the chance
of at least one spurious "significant" result is ~26%, and picking a winner out of six uncorrected
tests is exactly how a noise result gets into a thesis. Holm controls the family-wise error rate at
5% while staying more powerful than plain Bonferroni.

**One headline metric, decided in advance: Dice at the frozen `tau*`.** Ranking by whichever of the
three is most flattering is a garden-of-forking-paths problem. IoU is reported alongside as a
convention, *not* as confirmation -- it is algebraically the same measurement. Pointing Game
accuracy is the genuine second opinion, and the rank-agreement table checks whether the two agree.
If they disagree, that disagreement is the finding and should be reported, not resolved by picking
a favourite.

In [ ]:
from itertools import combinations
from scipy import stats

PRIMARY = "dice"      # at the fixed TAU -- decided in advance, see above


def series(store, arch, metric):
    """The per-image column for `metric` at the reported threshold."""
    return store[arch][col(metric, TAU)]


def holm(pvals):
    """Holm-Bonferroni step-down adjusted p-values. Uniformly more powerful than Bonferroni at the
    same family-wise error rate, so there is no reason to prefer the latter."""
    p = np.asarray(pvals, float)
    order = np.argsort(p)
    adj, running, m = np.empty_like(p), 0.0, len(p)
    for i, idx in enumerate(order):
        running = max(running, (m - i) * p[idx])
        adj[idx] = min(running, 1.0)
    return adj


def mcnemar_exact(x, y):
    """Exact McNemar for two paired binary vectors. Returns (x_only, y_only, p)."""
    x, y = np.asarray(x, bool), np.asarray(y, bool)
    n10, n01 = int((x & ~y).sum()), int((~x & y).sum())
    p = stats.binomtest(min(n01, n10), n01 + n10, 0.5).pvalue if (n01 + n10) else 1.0
    return n10, n01, float(p)


# --- ranking on the headline metric, with bootstrap CIs on all three ---
paired = pd.DataFrame({a: series(ALL_LOC, a, PRIMARY).values for a in ARCHS})
ranked = paired.mean().sort_values(ascending=False).index.tolist()

rank_rows = []
for arch in ranked:
    d = ALL_LOC[arch]
    iou, dice = series(ALL_LOC, arch, "iou"), series(ALL_LOC, arch, "dice")
    pga = d["pointing_hit"].astype(float)
    (il, ih), (dl, dh), (pl, ph) = boot_ci(iou), boot_ci(dice), boot_ci(pga)
    rank_rows.append({"arch": arch, "tau": TAU,
                      "IoU": iou.mean(), "iou_lo": il, "iou_hi": ih,
                      "Dice": dice.mean(), "dice_lo": dl, "dice_hi": dh,
                      "PGA_%": 100 * pga.mean(), "pga_lo_%": 100 * pl, "pga_hi_%": 100 * ph,
                      "conc_ratio_x": d["concentration_ratio"].mean(),
                      "peak_in_pad_%": 100 * d["peak_in_padding"].mean()})
rank_tbl = pd.DataFrame(rank_rows)
print(f"RANKED BY {PRIMARY.upper()} AT tau = {TAU:.2f}  (95% bootstrap CIs)")
print(rank_tbl.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# --- all 6 pairwise tests, on all three metrics, each family Holm-corrected on its own ---
pair_rows = {}
for metric in ("dice", "iou"):
    rows, raw = [], []
    for a, b in combinations(ranked, 2):
        va, vb = series(ALL_LOC, a, metric), series(ALL_LOC, b, metric)
        _, p = stats.wilcoxon(va, vb, nan_policy="omit")
        raw.append(p)
        rows.append({"metric": metric, "better": a, "worse": b,
                     "delta": va.mean() - vb.mean(), "win_rate_%": 100 * (va > vb).mean(),
                     "p_raw": p})
    t = pd.DataFrame(rows)
    t["p_holm"] = holm(raw)                      # corrected WITHIN the metric family, not across
    t["signif"] = np.where(t["p_holm"] < 0.05, "yes", "no")
    pair_rows[metric] = t

rows, raw = [], []
for a, b in combinations(ranked, 2):
    n10, n01, p = mcnemar_exact(ALL_LOC[a]["pointing_hit"], ALL_LOC[b]["pointing_hit"])
    raw.append(p)
    rows.append({"metric": "pointing", "better": a, "worse": b,
                 "delta": 100 * (ALL_LOC[a]["pointing_hit"].mean()
                                 - ALL_LOC[b]["pointing_hit"].mean()),
                 "a_only": n10, "b_only": n01, "p_raw": p})
t = pd.DataFrame(rows)
t["p_holm"] = holm(raw)
t["signif"] = np.where(t["p_holm"] < 0.05, "yes", "no")
pair_rows["pointing"] = t

print("\nPAIRWISE -- Wilcoxon signed-rank on Dice (Holm-corrected across all 6 tests)")
print(pair_rows["dice"].to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nPAIRWISE -- Wilcoxon signed-rank on IoU  (same ordering by construction; a convention "
      "check, not new evidence)")
print(pair_rows["iou"].to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nPAIRWISE -- McNemar exact on the Pointing Game  (delta in percentage points; a_only / "
      "b_only are the discordant images)")
print(pair_rows["pointing"].to_string(index=False, float_format=lambda x: f"{x:.4f}"))

pair_tbl = pd.concat(pair_rows.values(), ignore_index=True)

# --- does the winner survive a change of metric? ---
higher_better = {"Dice": True, "IoU": True, "pointing_hit": True, "pointing_hit_tol": True,
                 "concentration_ratio": True, "pixel_ap": True, "pixel_auroc": True,
                 "peak_dist_px": False}


def metric_series(arch, met):
    if met in ("Dice", "IoU"):
        return series(ALL_LOC, arch, met.lower())
    return ALL_LOC[arch][met]


agree = pd.DataFrame(
    {met: pd.Series({a: metric_series(a, met).mean() for a in ARCHS})
            .rank(ascending=not hi).astype(int)
     for met, hi in higher_better.items()})
agree["mean_rank"] = agree.mean(axis=1)
print("\nRANK BY METRIC (1 = best; mean_rank low = consistently good)")
print(agree.sort_values("mean_rank").to_string())

rank_tbl.to_csv(os.path.join(RESULTS_DIR, "interpretability_ranking.csv"), index=False)
pair_tbl.to_csv(os.path.join(RESULTS_DIR, "interpretability_pairwise.csv"), index=False)
agree.to_csv(os.path.join(RESULTS_DIR, "interpretability_rank_agreement.csv"))

In [ ]:
# --- is the most interpretable model also the most accurate? ---
# Accuracy comes from the training sweep, not recomputed here; same checkpoints, same test split.
runs = pd.read_csv(os.path.join(ROOT, "notebooks", "experiment_1_baseline", "results",
                                "hires_runs.csv"))
acc = runs.set_index("architecture")[["accuracy", "roc_auc"]]

verdict = rank_tbl.set_index("arch").join(acc)
assert not verdict[["accuracy", "roc_auc"]].isna().any().any(), \
    "accuracy join produced NaN -- an arch name in ARCHS is missing from hires_runs.csv"
verdict["interp_rank"] = verdict["Dice"].rank(ascending=False).astype(int)
verdict["acc_rank"] = verdict["roc_auc"].rank(ascending=False).astype(int)
print("INTERPRETABILITY vs CLASSIFICATION PERFORMANCE")
print(verdict[["IoU", "Dice", "PGA_%", "interp_rank", "accuracy", "roc_auc", "acc_rank"]]
      .to_string(float_format=lambda x: f"{x:.4f}"))

# --- the verdict, stated rather than left to the reader ---
top = ranked[0]
dice_beats = pair_rows["dice"].query("better == @top and signif == 'yes'")["worse"].tolist()
pga_top = max(ARCHS, key=lambda a: ALL_LOC[a]["pointing_hit"].mean())
pga_beats = pair_rows["pointing"].query("better == @pga_top and signif == 'yes'")["worse"].tolist()

print(f"\n{'=' * 84}")
print(f"BEST BY DICE / IoU : {top}  "
      f"(Dice {verdict.loc[top, 'Dice']:.4f} [{verdict.loc[top, 'dice_lo']:.4f}, "
      f"{verdict.loc[top, 'dice_hi']:.4f}], IoU {verdict.loc[top, 'IoU']:.4f} "
      f"[{verdict.loc[top, 'iou_lo']:.4f}, {verdict.loc[top, 'iou_hi']:.4f}])")
if dice_beats:
    print(f"  significantly beats (Wilcoxon, Holm): {', '.join(dice_beats)}")
    ties = [a for a in ranked[1:] if a not in dice_beats]
    if ties:
        print(f"  NOT separable from: {', '.join(ties)} -- report these as tied, not ranked")
else:
    print("  but beats NO other architecture significantly after correction -- the correct claim")
    print("  is that the four are statistically indistinguishable on overlap.")

print(f"\nBEST BY POINTING GAME : {pga_top}  "
      f"({verdict.loc[pga_top, 'PGA_%']:.1f}% [{verdict.loc[pga_top, 'pga_lo_%']:.1f}, "
      f"{verdict.loc[pga_top, 'pga_hi_%']:.1f}])")
if pga_beats:
    print(f"  significantly beats (McNemar, Holm): {', '.join(pga_beats)}")
else:
    print("  but beats NO other architecture significantly after correction.")

if top == pga_top:
    print(f"\nThe two independent metrics agree on {top}. That agreement is the strongest single")
    print("sentence available here, because Dice and the pointing game can fail in opposite ways:")
    print("a big sloppy blob wins Dice while missing the peak; a lucky peak wins the pointing game")
    print("with no overlap at all.")
else:
    print(f"\nThe metrics DISAGREE: Dice favours {top}, the pointing game favours {pga_top}.")
    print("Report the disagreement -- it means one backbone covers the lesion more completely")
    print("while the other aims its peak more accurately. Do not resolve it by picking one.")

srho = stats.spearmanr(verdict["Dice"], verdict["roc_auc"]).statistic
print(f"\nSpearman(Dice, ROC AUC) = {srho:.2f} over n=4 architectures -- far too few points for")
print("this to be evidence either way; report it as an observation, not a trend.")
print("=" * 84)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8.2))
(ax_iou, ax_dice), (ax_pga, ax_sc) = axes


def ranked_bars(ax, value_col, lo_col, hi_col, title, xlabel, chance_line=None, fmt="{:.3f}"):
    """One metric, one panel, one axis. Small multiples rather than a grouped bar chart: grouping
    would need a second colour dimension, and colour here is bound to the ARCHITECTURE so a
    backbone keeps its hue in every figure."""
    order = verdict.sort_values(value_col).index.tolist()      # best at top of a horizontal bar
    y = np.arange(len(order))
    mean = verdict.loc[order, value_col].to_numpy(float)
    lo = mean - verdict.loc[order, lo_col].to_numpy(float)
    hi = verdict.loc[order, hi_col].to_numpy(float) - mean

    ax.barh(y, mean, height=0.62, color=[ARCH_COLOR[a] for a in order])
    ax.errorbar(mean, y, xerr=[lo, hi], fmt="none", ecolor=INK, elinewidth=1.4, capsize=4)
    if chance_line is not None:
        ax.axvline(chance_line, color=MUTED, ls="--", lw=1.2)
        ax.text(chance_line, len(order) - 0.34, " chance", color=MUTED, fontsize=8, va="center")
    for yi, v in enumerate(mean):        # direct labels -- value never rests on colour alone
        ax.text(v + hi[yi] + mean.max() * 0.035, yi, fmt.format(v), va="center", fontsize=9,
                color=INK)
    ax.set_yticks(y, order, fontsize=9)
    ax.set_xlabel(xlabel, color=MUTED, fontsize=9)
    ax.set_title(title, fontsize=10, loc="left")
    ax.set_xlim(0, (mean + hi).max() * 1.30)


ranked_bars(ax_iou, "IoU", "iou_lo", "iou_hi",
            "IoU at tau = 0.50  (95% CI)", "IoU")
ranked_bars(ax_dice, "Dice", "dice_lo", "dice_hi",
            "Dice at tau = 0.50  (95% CI)", "Dice coefficient")
ranked_bars(ax_pga, "PGA_%", "pga_lo_%", "pga_hi_%",
            "Pointing Game accuracy  (95% CI)", "% of test images with the peak inside the ROI",
            chance_line=CHANCE_PGA, fmt="{:.1f}%")

# --- interpretability vs accuracy: two measures on ONE scatter, never a dual y-axis ---
for a in ARCHS:
    ax_sc.scatter(verdict.loc[a, "roc_auc"], verdict.loc[a, "Dice"], s=110, color=ARCH_COLOR[a],
                  zorder=3, edgecolor="white", linewidth=1.5)
    ax_sc.annotate(a, (verdict.loc[a, "roc_auc"], verdict.loc[a, "Dice"]),
                   textcoords="offset points", xytext=(9, 0), fontsize=8.5, color=INK, va="center")
ax_sc.set_xlabel("test ROC AUC (classification)", color=MUTED, fontsize=9)
ax_sc.set_ylabel("Dice (explanation)", color=MUTED, fontsize=9)
ax_sc.set_title("Does accuracy buy interpretability?", fontsize=10, loc="left")
ax_sc.margins(x=0.26, y=0.22)

for ax in axes.ravel():
    ax.grid(axis="both" if ax is ax_sc else "x", color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(GRID)
    ax.tick_params(colors=MUTED, labelsize=8.5)
for ax in (ax_iou, ax_dice, ax_pga):
    ax.tick_params(axis="y", colors=INK, labelsize=9)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "interpretability_ranking.png"), dpi=150, bbox_inches="tight")
plt.show()

## 7. Control vs seg-aux (paired)

Section 6 asks which *backbone* explains best. This section asks a different question: for a fixed
backbone, does **auxiliary ROI-mask supervision produce better explanations**? Same paired logic,
same tests -- McNemar's exact for the binary pointing game, Wilcoxon signed-rank for the continuous
metrics.

**Two fairness rules, or the comparison is worthless:**

1. The seg-aux model's CAM must come from the **classifier** path -- gradients of the class-1 logit
   w.r.t. `extractor` -- *not* from the segmentation decoder's output. Otherwise you are comparing a
   segmentation model's segmentation against a classifier's Grad-CAM, which is not the claim. The
   decoder output may be reported separately as a clearly labelled auxiliary-localization row.
2. Both models must share architecture, resolution, batch size, epoch budget, seed and pooling.
   Only the seg loss term differs.

No hi-res seg-aux checkpoint exists yet, so this no-ops. When they are trained, run it once per
architecture -- whether seg-aux helps is itself likely to be backbone-dependent, and a single
architecture cannot answer it.

In [ ]:
COMPARE = {
    # "control": "<path to hi-res control checkpoint>",
    # "segaux":  "<path to hi-res segaux checkpoint>",
}

if len(COMPARE) < 2:
    print("Set COMPARE to two hi-res checkpoints (same architecture) to run the paired comparison.")
else:
    results, archs_seen = {}, set()
    for name, path in COMPARE.items():
        ck = torch.load(path, map_location=H.DEVICE, weights_only=False)
        assert list(ck.get("input_size", [])) == [H.HIRES_H, H.HIRES_W], f"{name}: wrong input_size"
        # architecture comes from the checkpoint -- hardcoding it here would silently compare a
        # control of one backbone against a segaux of another
        arch_name = ck["arch"]
        archs_seen.add(arch_name)
        m = H.SingleInputModel(arch_name, pooling=ck.get("pooling", "gap"),
                               use_seg=ck.get("use_seg", False)).to(H.DEVICE)
        m.load_state_dict(ck["model_state_dict"])
        m.eval()
        with H.GradCAM(m) as cf:            # classifier path, NOT the seg decoder
            results[name] = evaluate_localization(test_df, cf, desc=f"{name} test ({arch_name})")
        del m
        H.free_gpu()
    assert len(archs_seen) == 1, f"COMPARE mixes architectures {archs_seen} -- not a fair pairing"

    a, b = list(COMPARE)
    da, db = results[a], results[b]
    assert da["row_idx"].tolist() == db["row_idx"].tolist(), "row sets differ -- not paired"
    print(pd.DataFrame([summarize(da, a, TAU), summarize(db, b, TAU)])
          .to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    # Pointing Game: paired binary -> McNemar exact
    n10, n01, p_mc = mcnemar_exact(da["pointing_hit"], db["pointing_hit"])
    print(f"\nMcNemar (Pointing Game): {a} wins {n10}, {b} wins {n01}, p = {p_mc:.4f}")

    # IoU / Dice: paired continuous -> Wilcoxon signed-rank, both at the same fixed tau
    for metric in ("iou", "dice"):
        va, vb = da[col(metric, TAU)], db[col(metric, TAU)]
        _, p = stats.wilcoxon(va, vb, nan_policy="omit")
        print(f"Wilcoxon {metric.upper():5s}: {a} {va.mean():.4f} vs {b} {vb.mean():.4f}   "
              f"p = {p:.4f}")
    for metric in ("energy_concentration", "concentration_ratio", "pixel_ap"):
        _, p = stats.wilcoxon(da[metric], db[metric], nan_policy="omit")
        print(f"Wilcoxon {metric:22s}: {a} {da[metric].mean():.4f} vs "
              f"{b} {db[metric].mean():.4f}   p = {p:.4f}")

## 8. What to record for the thesis

**Quote all three metrics together, per architecture, with CIs.** IoU, Dice and Pointing Game
accuracy answer different questions and the table is only honest with all three present. The
one-sentence version to lead with is the pointing game, because it needs no threshold:

> *"On N% of test mammograms the hottest point of the Grad-CAM explanation falls inside the
> radiologist-drawn ROI, against a 2.5% chance rate."*

- **State the threshold every time IoU or Dice appears.** "Grad-CAM thresholded at `tau = 0.50`,
  fixed a priori" is one clause and it forecloses the obvious objection. Cite the appendix sweep
  for the claim that the ranking does not depend on it.
- **Say plainly that Dice and IoU are one measurement.** `Dice = 2*IoU/(1+IoU)`. Reporting both is
  convention; presenting them as two agreeing metrics would be double-counting. Pointing Game
  accuracy is the independent one.
- **Report the IoU/Dice curve, not only the single value.** With a target under 1% of the image and
  a 20x12 CAM grid, absolute overlap is low by construction; a lone number invites comparison
  against models that were actually trained to segment. Pair it with `cam_grid` from section 2.
- **Compare each architecture against its OWN random-init floor**, not against zero. Step 3 prints
  the lift; a backbone sitting near its own floor is not localizing, and the headline table alone
  will not show that.
- **Report all four architectures, not the winner alone.** The ranking plus the Holm-corrected
  pairwise tests are the result. "X localizes best" is only a claim if it survives correction;
  otherwise the honest sentence is *"the four are statistically indistinguishable on
  localization"*, which is itself worth stating.
- **If Dice and the pointing game disagree on the winner, report the disagreement.** It means one
  backbone covers the lesion more completely while another aims its peak better -- a real finding,
  not a problem to resolve by choosing the flattering metric.
- **Report the peak-in-padding rate even when it is near zero** -- it demonstrates the model is not
  exploiting the canvas geometry.
- **Interpretability and accuracy are separate axes.** If the most interpretable backbone is not the
  most accurate, say so plainly -- with n=4 architectures there is no statistical basis for calling
  it a trade-off, but it is a legitimate observation and a reason to pick a deployment model on
  something other than AUC alone.
- If seg-aux improves localization but not accuracy, **report exactly that**. Explainability gained
  at no accuracy cost is a real, publishable result; do not reshape it into an accuracy story.

In [ ]:
# =====================================================================================
#  FINAL -- the three interpretability metrics
# =====================================================================================
# Everything above is the harness that makes these three numbers trustworthy: the geometry
# round-trips (s1), CAM peaks track a known input patch (s3b), a random model scores at chance
# (s3a), the fast sweep matches the naive one (s5.1), and the ranking survives every threshold
# (appendix) and Holm correction (s6). This is the table that goes in the thesis.
FINAL = pd.DataFrame([{
    "Architecture": H.ARCH_DISPLAY_NAMES.get(a, a),
    "IoU": ALL_LOC[a][col("iou", TAU)].mean(),
    "Dice": ALL_LOC[a][col("dice", TAU)].mean(),
    "PointingGameAcc_%": 100 * ALL_LOC[a]["pointing_hit"].mean(),
} for a in ARCHS]).sort_values("Dice", ascending=False).reset_index(drop=True)

print(f"INTERPRETABILITY -- CBIS-DDSM test split, n = {len(ALL_LOC[ARCHS[0]])} images")
print(f"Grad-CAM vs radiologist ROI, heatmap thresholded at tau = {TAU:.2f}\n")
print(FINAL.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
FINAL.to_csv(os.path.join(RESULTS_DIR, "interpretability_three_metrics.csv"), index=False)

# Which architecture tops each metric -- computed, not asserted.
tops = {m: FINAL.loc[FINAL[m].idxmax(), "Architecture"]
        for m in ("IoU", "Dice", "PointingGameAcc_%")}
best = FINAL.iloc[0]
print(f"\nchance floor: pointing accuracy {CHANCE_PGA:.2f}% (the lesion's share of the breast)")
if len(set(tops.values())) == 1:
    print(f"\n{best['Architecture']} is best on all three metrics: IoU {best['IoU']:.3f}, "
          f"Dice {best['Dice']:.3f}, pointing game {best['PointingGameAcc_%']:.1f}% "
          f"({best['PointingGameAcc_%'] / CHANCE_PGA:.1f}x chance).")
else:
    print(f"\nThe metrics disagree -- report the disagreement rather than picking one: {tops}")